# Diffusion Kernel Pipeline — Development Notebook

This notebook develops and tests the two modifications to the spectral clustering pipeline:

1. **k-NN affinity** — replaces the fully connected Gaussian kernel with a symmetric k-nearest-neighbour graph to enforce manifold-respecting sparsity.
2. **Diffusion map re-weighting** — scales spectral embedding coordinates by diffusion eigenvalues raised to power *t* for robustness under rank over-specification.

Both modifications are backward-compatible: `k = n − 1` reproduces the fully connected affinity exactly, and `t = 0` applies no re-weighting.

---
## Step 4.1 — k-NN Affinity Matrix Construction

In [1]:
import sys
import os

# Add the project root to the Python path so local modules can be imported
# from any working directory.
project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

import numpy as np
import matplotlib.pyplot as plt

from clustering.spectral import dist_mat, knn_affinity
from data.generators import make_blobs_convex, make_rings, make_two_moons

### Dataset generation

We use three standard synthetic datasets from the dissertation experiments:

| Dataset | Generator | σ |
|---------|-----------|---|
| Seven Gaussian blobs | `make_blobs_convex` | 25 |
| Three concentric circles | `make_rings` | 0.5 |
| Two interlocking moons | `make_two_moons` | 0.5 |

In [2]:
SEED = 0

# Seven Gaussian blobs (2-D for visualisation; high-D used in experiments)
X_blobs, y_blobs = make_blobs_convex(N=1200, num_clusters=7, n_features=5000, seed=SEED)
sigma_blobs = 10.0

# Three concentric circles
X_rings, y_rings = make_rings(n_points=1200, n_circles=3, noise_std=0.05, seed=SEED)
sigma_rings = 0.5

# Two interlocking moons
X_moons, y_moons = make_two_moons(n_samples=1200, noise=0.05, seed=SEED)
sigma_moons = 0.1

datasets = [
    ("Blobs",   X_blobs, y_blobs, sigma_blobs),
    ("Circles", X_rings, y_rings, sigma_rings),
    ("Moons",   X_moons, y_moons, sigma_moons),
]

print("Dataset sizes:")
for name, X, y, sigma in datasets:
    print(f"  {name:8s}: n={X.shape[0]}, d={X.shape[1]}, sigma={sigma}")

Dataset sizes:
  Blobs   : n=1200, d=5000, sigma=10.0
  Circles : n=1200, d=2, sigma=0.5
  Moons   : n=1200, d=2, sigma=0.1


---
### Test 1 — Backward Compatibility (`k = n − 1`)

When `k >= n − 1`, `knn_affinity` must return exactly the same matrix as the
fully connected Gaussian affinity used in `KM_spec_decom` (diagonal zeroed).

In [ ]:
print("Test 1 — Backward compatibility (k = n - 1)")
print("=" * 50)

all_passed = True

for name, X, y, sigma in datasets:
    n = X.shape[0]
    D = dist_mat(X)

    # knn_affinity with k = n - 1 triggers the fully connected fallback
    A_knn = knn_affinity(D, k=n-1, sigma=sigma)

    # Reference: manually construct the fully connected Gaussian affinity
    A_full = np.exp(-D ** 2 / (2.0 * sigma ** 2))
    np.fill_diagonal(A_full, 0.0)

    passed = np.allclose(A_knn, A_full)
    all_passed = all_passed and passed
    status = "PASS" if passed else "FAIL"
    print(f"  {name:8s} (n={n}): {status}")
    if not passed:
        max_diff = np.max(np.abs(A_knn - A_full))
        print(f"             max |A_knn - A_full| = {max_diff:.2e}")

print()
print("Overall:", "ALL PASSED" if all_passed else "FAILURES DETECTED")

---
### Test 2 — Sparsity Visualisation (`k = 10`)

The k-NN affinity matrix (k = 10) should be visibly sparser than the fully
connected affinity.  Points are sorted by their true label so that intra-cluster
structure appears as blocks along the diagonal.

In [ ]:
K_VIS = 10

fig, axes = plt.subplots(
    nrows=3, ncols=2,
    figsize=(10, 13),
    constrained_layout=True,
)

fig.suptitle(
    f"Affinity matrices — fully connected (left) vs k-NN k={K_VIS} (right)",
    fontsize=13,
)

for row_idx, (name, X, y, sigma) in enumerate(datasets):
    D = dist_mat(X)

    # Fully connected reference
    A_full = np.exp(-D ** 2 / (2.0 * sigma ** 2))
    np.fill_diagonal(A_full, 0.0)

    # k-NN affinity
    A_knn = knn_affinity(D, k=K_VIS, sigma=sigma)

    # Sort points by true label so cluster structure is visible as blocks
    order = np.argsort(y)

    ax_full = axes[row_idx, 0]
    ax_knn  = axes[row_idx, 1]

    im0 = ax_full.imshow(
        A_full[np.ix_(order, order)], aspect="auto", cmap="viridis"
    )
    ax_full.set_title(f"{name} — fully connected (sigma={sigma})", fontsize=10)
    ax_full.set_xlabel("Point index (sorted by label)")
    ax_full.set_ylabel("Point index (sorted by label)")
    plt.colorbar(im0, ax=ax_full, fraction=0.046, pad=0.04)

    im1 = ax_knn.imshow(
        A_knn[np.ix_(order, order)], aspect="auto", cmap="viridis"
    )
    ax_knn.set_title(f"{name} — k-NN k={K_VIS} (sigma={sigma})", fontsize=10)
    ax_knn.set_xlabel("Point index (sorted by label)")
    ax_knn.set_ylabel("Point index (sorted by label)")
    plt.colorbar(im1, ax=ax_knn, fraction=0.046, pad=0.04)

plt.show()

---
### Test 3 — Sanity Checks on Matrix Properties (`k = 10`)

For each k-NN affinity matrix we assert:

1. **Symmetry** — `A == A.T` to numerical precision.
2. **Non-negativity** — all entries ≥ 0.
3. **Zero diagonal** — self-affinities are zeroed.
4. **Sparsity bounds** — each row has between `k` and `2k` non-zero entries.
   Lower bound: at least the k directed neighbours.  Upper bound: own k
   neighbours plus at most k additional reverse-neighbour edges added by
   the OR-symmetrisation.

In [ ]:
K_CHECK = 10

print(f"Test 3 — Matrix property checks (k = {K_CHECK})")
print("=" * 60)

all_passed = True

for name, X, y, sigma in datasets:
    n = X.shape[0]
    D = dist_mat(X)
    A = knn_affinity(D, k=K_CHECK, sigma=sigma)

    results = {}

    # 1. Symmetry
    results["Symmetric"] = bool(np.allclose(A, A.T))

    # 2. Non-negativity
    results["Non-negative"] = bool((A >= 0.0).all())

    # 3. Zero diagonal
    results["Zero diagonal"] = bool(np.allclose(np.diag(A), 0.0))

    # 4. Sparsity: non-zero entries per row must lie in [k, 2*k]
    nnz_per_row = (A > 0.0).sum(axis=1)  # (n,) integer array
    results["Sparsity bounds"] = bool(
        (nnz_per_row >= K_CHECK).all() and (nnz_per_row <= 2 * K_CHECK).all()
    )

    dataset_passed = all(results.values())
    all_passed = all_passed and dataset_passed

    print(f"\n  {name} (n={n}, sigma={sigma}):")
    for check, ok in results.items():
        status = "PASS" if ok else "FAIL"
        print(f"    [{status}] {check}")

    # Report sparsity statistics for reference
    print(f"    Non-zero entries per row: "
          f"min={nnz_per_row.min()}, "
          f"max={nnz_per_row.max()}, "
          f"mean={nnz_per_row.mean():.1f}")

print()
print("Overall:", "ALL PASSED" if all_passed else "FAILURES DETECTED")

---
## Step 4.2 — Generalised Spectral Embedding with Diffusion Weighting

`diffusion_spectral_embed` extends `spectral_embed` with two new parameters:

| Parameter | Meaning | Backward-compatible default |
|-----------|---------|----------------------------|
| `k_neighbours` | k-NN neighbourhood size | `n - 1` (fully connected) |
| `t` | diffusion time | `0` (no re-weighting) |

Setting `k_neighbours = n - 1` and `t = 0` reproduces `spectral_embed` exactly.

### Implementation

In [3]:
import scipy.linalg

def diffusion_spectral_embed(
    X,
    sigma,
    k_neighbours,
    n_clusters,
    q=None,
    Laplacian="RW",
    t=0,
    seed=6,
):
    """
    Generalised spectral embedding with optional k-NN sparsification and
    diffusion map eigenvalue re-weighting.

    Parameters
    ----------
    X : (n, d) ndarray
        Input data points.
    sigma : float
        Gaussian kernel bandwidth.
    k_neighbours : int
        Number of nearest neighbours for graph construction.  When
        k_neighbours >= n - 1 the graph is fully connected, reproducing
        the existing pipeline exactly (including the unzeroed diagonal of
        the affinity matrix, which matches KM_spec_decom).
    n_clusters : int
        Target number of clusters; used as the default embedding dimension
        when q is None.
    q : int or None
        Number of eigenvectors to retain (embedding dimension).
        Defaults to n_clusters.
    Laplacian : {"RW", "SYM"}
        Random-walk or symmetric normalised Laplacian.
    t : float
        Diffusion time.  When t = 0 no re-weighting is applied.  When
        t > 0 eigenvector coordinates are scaled by mu_j^t where
        mu_j = 1 - lambda_j are the transition-matrix eigenvalues.
    seed : int
        Random seed for reproducibility.

    Returns
    -------
    Y : (n, q_eff) ndarray
        Spectral (or diffusion) embedding.
    eig_vals : (n,) ndarray
        Full eigenvalue array in ascending order, for diagnostics.

    Notes
    -----
    Backward-compatibility is exact when k_neighbours >= n - 1 and t = 0.
    In the fully connected fallback the affinity diagonal is left as
    exp(0) = 1 (not zeroed), which replicates KM_spec_decom exactly and
    ensures the degree computation and Laplacian are identical.  For sparse
    k-NN graphs the diagonal is always zeroed by knn_affinity.
    """
    np.random.seed(seed)
    n = X.shape[0]
    I = np.eye(n)

    # Step 1 — Distance matrix.
    dist_matrix = dist_mat(X)

    # Step 2 — Affinity matrix.
    # For the fully connected fallback we reproduce KM_spec_decom exactly:
    # the diagonal is NOT zeroed (exp(-0) = 1), matching the original degree
    # computation.  For sparse graphs knn_affinity zeros the diagonal.
    if k_neighbours >= n - 1:
        A = np.exp(-dist_matrix ** 2 / (2.0 * sigma ** 2))
    else:
        A = knn_affinity(dist_matrix, k=k_neighbours, sigma=sigma)

    # Step 3 — Degree matrix and normalised Laplacian.
    d = np.sum(A, axis=1)

    if Laplacian.upper() == "RW":
        D_inv = np.diag(1.0 / d)
        L = I - D_inv @ A
        eig_vals, eig_vecs = scipy.linalg.eigh(L)

    elif Laplacian.upper() == "SYM":
        D_inv_sqrt = np.diag(1.0 / np.sqrt(d))
        L_sym = I - D_inv_sqrt @ A @ D_inv_sqrt
        eig_vals, eig_vecs = scipy.linalg.eigh(L_sym)

    else:
        raise ValueError("Laplacian must be 'RW' or 'SYM'.")

    # Step 4 — Select leading eigenvectors.
    # scipy.linalg.eigh returns eigenvalues in ascending order, so the
    # smallest (cluster-informative) eigenvalues are already first.
    q_eff = q if q is not None else n_clusters
    U = eig_vecs[:, :q_eff]          # (n, q_eff)
    lambdas = eig_vals[:q_eff]        # (q_eff,)

    # Step 5 — Row normalisation (Ng–Jordan–Weiss, SYM only).
    if Laplacian.upper() == "SYM":
        row_norms = np.sqrt(np.sum(U ** 2, axis=1, keepdims=True))
        U = U / np.maximum(row_norms, 1e-12)

    # Step 6 — Diffusion map eigenvalue re-weighting.
    # Convert Laplacian eigenvalues to transition-matrix eigenvalues.
    mu = 1.0 - lambdas          # mu[0] ≈ 1 (trivial mode); decreasing
    weights = mu ** t            # all ones when t = 0
    Y = U * weights[np.newaxis, :]   # broadcast across rows

    # Step 7 — Return embedding and full eigenvalue array for diagnostics.
    return Y, eig_vals

---
### Test 1 — Backward Compatibility Against `spectral_embed`

With `k_neighbours = n - 1` and `t = 0`, `diffusion_spectral_embed` must
produce an embedding numerically identical to `spectral_embed`.

> **Note on eigenvector sign ambiguity.** Eigenvectors are defined only up
> to a global sign flip, so columns of *Y* may differ by ±1. The test
> checks exact match first; if that fails it checks absolute values. Agreement
> on absolute values confirms a correct implementation, not a bug.

In [4]:
from clustering.spectral import spectral_embed

SEED_T1 = 6   # default seed used by spectral_embed

test_configs = [
    # (name,    X,        y,        sigma,       n_clusters, q)
    ("Blobs",   X_blobs,  y_blobs,  sigma_blobs, 7,          7),
    ("Circles", X_rings,  y_rings,  sigma_rings, 3,          3),
    ("Moons",   X_moons,  y_moons,  sigma_moons, 2,          2),
]

print("Test 1 — Backward compatibility (k_neighbours = n-1, t = 0)")
print("=" * 65)

all_passed = True

for lap in ("RW", "SYM"):
    print(f"\n  Laplacian = {lap}")
    for name, X, y, sigma, n_clusters, q in test_configs:
        n = X.shape[0]

        # Existing function
        Y_old, evals_old = spectral_embed(
            X, sigma=sigma, k=n_clusters, q=q, Laplacian=lap, seed=SEED_T1
        )

        # New function with fully connected fallback and no diffusion weighting
        Y_new, evals_new = diffusion_spectral_embed(
            X, sigma=sigma, k_neighbours=n - 1, n_clusters=n_clusters,
            q=q, Laplacian=lap, t=0, seed=SEED_T1
        )

        # Eigenvalues are not sign-ambiguous
        evals_ok = np.allclose(evals_old[:q], evals_new[:q], atol=1e-10)

        # Eigenvectors are defined up to a sign flip per column — check both
        embed_exact = np.allclose(Y_old, Y_new, atol=1e-10)
        embed_abs   = np.allclose(np.abs(Y_old), np.abs(Y_new), atol=1e-10)
        embed_ok    = embed_exact or embed_abs
        sign_note   = "" if embed_exact else " (sign flip — expected)"

        passed = evals_ok and embed_ok
        all_passed = all_passed and passed
        status = "PASS" if passed else "FAIL"

        print(f"    [{status}] {name:8s}  "
              f"evals={'ok' if evals_ok else 'FAIL'}  "
              f"embed={'ok' if embed_ok else 'FAIL'}{sign_note}")

print()
print("Overall:", "ALL PASSED" if all_passed else "FAILURES DETECTED")

Test 1 — Backward compatibility (k_neighbours = n-1, t = 0)

  Laplacian = RW
    [PASS] Blobs     evals=ok  embed=ok
    [PASS] Circles   evals=ok  embed=ok
    [PASS] Moons     evals=ok  embed=ok

  Laplacian = SYM
    [PASS] Blobs     evals=ok  embed=ok
    [PASS] Circles   evals=ok  embed=ok
    [PASS] Moons     evals=ok  embed=ok

Overall: ALL PASSED


---
### Test 2 — Diffusion Weighting Effect

As *t* increases, later eigenvector columns (corresponding to smaller
transition-matrix eigenvalues μ_j) should be progressively suppressed
relative to the first column. The first column's weight (μ₀^t) should
remain close to 1 throughout because μ₀ ≈ 1.

In [ ]:
# Two moons, k-NN graph, tighter bandwidth to expose cluster structure
sigma_dm  = 0.1
k_dm      = 10
q_dm      = 5
lap_dm    = "RW"
t_values  = [0, 1, 5, 20]

print("Test 2 — Diffusion weighting effect")
print(f"  Dataset: Two Moons  |  sigma={sigma_dm}  |  k={k_dm}  |  q={q_dm}  |  Laplacian={lap_dm}")
print("=" * 70)

for t in t_values:
    Y, eig_vals = diffusion_spectral_embed(
        X_moons, sigma=sigma_dm, k_neighbours=k_dm,
        n_clusters=2, q=q_dm, Laplacian=lap_dm, t=t, seed=SEED,
    )

    lambdas   = eig_vals[:q_dm]
    mu        = 1.0 - lambdas
    weights   = mu ** t
    col_norms = np.linalg.norm(Y, axis=0)

    print(f"\n  t = {t}")
    print(f"    mu (transition eigenvalues):   {np.array2string(mu, precision=4, suppress_small=True)}")
    print(f"    weights (mu^t):                {np.array2string(weights, precision=4, suppress_small=True)}")
    print(f"    column norms of Y:             {np.array2string(col_norms, precision=4, suppress_small=True)}")

    if t == 0:
        ok = np.allclose(weights, 1.0)
        print(f"    [{'PASS' if ok else 'FAIL'}] All weights = 1 at t=0")
    else:
        ratio     = col_norms[1:] / (col_norms[0] + 1e-12)
        monotone  = bool((np.diff(ratio) <= 0).all())
        print(f"    norm ratios col[1:]/col[0]: {np.array2string(ratio, precision=4, suppress_small=True)}")
        print(f"    [{'PASS' if monotone else 'FAIL'}] Norms decrease monotonically with eigenvector index")

---
### Test 3 — Visual Comparison of Embeddings at Varying *t*

The 2-D spectral embedding (columns 0 and 1 of *Y*) for the two moons
dataset at four diffusion times. As *t* grows the dominant spectral mode
(moon separator) should be emphasised relative to within-moon variation.

In [ ]:
sigma_vis = 0.1
k_vis     = 10
q_vis     = 2
lap_vis   = "RW"
t_values_vis = [0, 1, 5, 10]

fig, axes = plt.subplots(2, 2, figsize=(10, 8), constrained_layout=True)
fig.suptitle(
    "Two Moons — 2-D spectral embedding at varying diffusion time t\n"
    f"(sigma={sigma_vis}, k={k_vis}, Laplacian={lap_vis})",
    fontsize=12,
)

colours = plt.cm.tab10(np.linspace(0, 0.2, 2))

for ax, t in zip(axes.ravel(), t_values_vis):
    Y, _ = diffusion_spectral_embed(
        X_moons, sigma=sigma_vis, k_neighbours=k_vis,
        n_clusters=2, q=q_vis, Laplacian=lap_vis, t=t, seed=SEED,
    )
    for label in np.unique(y_moons):
        mask = y_moons == label
        ax.scatter(Y[mask, 0], Y[mask, 1], s=8, alpha=0.7,
                   color=colours[label], label=f"Moon {label}")
    ax.set_title(f"t = {t}", fontsize=11)
    ax.set_xlabel("Eigenvector 1")
    ax.set_ylabel("Eigenvector 2")
    ax.legend(markerscale=2, fontsize=8)

plt.show()

---
### Test 4 — ALC Clustering Sanity Check

Compare ALC clustering performance on two moons between:

- **Config A** — current pipeline equivalent: fully connected affinity, no diffusion weighting.
- **Config B** — new pipeline: k-NN affinity (k = 10) with diffusion weighting (t = 5).

Config B should produce a meaningfully higher Adjusted Rand Index (ARI)
against the ground truth labels. This is the core empirical claim of the
new pipeline on non-convex geometries.

In [ ]:
from clustering.alc import ALCAdapter
from sklearn.metrics import adjusted_rand_score

alc = ALCAdapter()

n_moons = X_moons.shape[0]

# Config A — fully connected affinity, no diffusion weighting (current pipeline)
Y_A, _ = diffusion_spectral_embed(
    X_moons, sigma=0.1, k_neighbours=n_moons - 1, n_clusters=2,
    q=2, Laplacian="RW", t=0, seed=SEED,
)
np.random.seed(42)
labels_A = alc.fit_predict(Y_A, cn=None)
ari_A = adjusted_rand_score(y_moons, labels_A)

# Config B — k-NN sparsification (k=10) plus diffusion weighting (t=5)
Y_B, _ = diffusion_spectral_embed(
    X_moons, sigma=0.1, k_neighbours=10, n_clusters=2,
    q=2, Laplacian="RW", t=5, seed=SEED,
)
np.random.seed(42)
labels_B = alc.fit_predict(Y_B, cn=None)
ari_B = adjusted_rand_score(y_moons, labels_B)

print("Test 4 — ALC clustering ARI on Two Moons")
print("=" * 45)
print(f"  Config A (fully connected, t=0):   ARI = {ari_A:.4f}")
print(f"  Config B (k-NN k=10, t=5):         ARI = {ari_B:.4f}")
print()
if ari_B > ari_A:
    print(f"  [PASS] Config B improves ARI by {ari_B - ari_A:+.4f}")
else:
    print(f"  [NOTE] Config B did not improve ARI (delta = {ari_B - ari_A:+.4f})")
    print("         Try adjusting sigma, k, or t for this dataset/seed.")

---

## Step 4.3 — Exploratory Runs: 2 × 2 Factorial Design

Four configurations isolate the individual contributions of k-NN
sparsification and diffusion eigenvalue weighting:

| Config | Graph | Diffusion | Role |
|--------|-------|-----------|------|
| **A** | Fully connected | t = 0 | Baseline — current pipeline |
| **C** | Fully connected | t > 0 | Diffusion effect only |
| **D** | k-NN sparse      | t = 0 | k-NN effect only |
| **B** | k-NN sparse      | t > 0 | Full new pipeline |

**Key comparisons in Plot 1:**

- **D vs A** (red solid vs blue solid) at r = q: marginal value of k-NN
  sparsification alone.
- **C vs A** (blue dashed vs blue solid) at r = q and r = 25: marginal
  value of diffusion weighting on a fully connected graph.
- **B vs D** (red dashed vs red solid) at **r = 25**: marginal value of
  adding diffusion *on top of* k-NN under rank over-specification. This
  is the critical comparison.

In [ ]:
import pandas as pd
from sklearn.metrics import adjusted_rand_score

# Datasets are already defined in Step 4.1; reference them directly.
# Circles are stored as X_rings / y_rings in this notebook.
EMBED_SEED = 6

n_blobs = X_blobs.shape[0]
n_rings = X_rings.shape[0]
n_moons = X_moons.shape[0]

experiment_datasets = [
    {
        "name":       "blobs",
        "X":          X_blobs,
        "y_true":     y_blobs,
        "n_clusters": 7,
        "r_values":   [7, 9, 12, 17, 25, 50],
        "configs": {
            "A_full_noDiff": {"sigma": 10,  "k_neighbours": n_blobs - 1, "t": 0},
            "C_full_Diff":   {"sigma": 10,  "k_neighbours": n_blobs - 1, "t": 3},
            "D_knn_noDiff":  {"sigma": 10,  "k_neighbours": 15,          "t": 0},
            "B_knn_Diff":    {"sigma": 10,  "k_neighbours": 15,          "t": 3},
        },
    },
    {
        "name":       "circles",
        "X":          X_rings,
        "y_true":     y_rings,
        "n_clusters": 3,
        "r_values":   [3, 5, 8, 13, 25, 50],
        "configs": {
            "A_full_noDiff": {"sigma": 0.5, "k_neighbours": n_rings - 1, "t": 0},
            "C_full_Diff":   {"sigma": 0.5, "k_neighbours": n_rings - 1, "t": 5},
            "D_knn_noDiff":  {"sigma": 0.5, "k_neighbours": 10,          "t": 0},
            "B_knn_Diff":    {"sigma": 0.5, "k_neighbours": 10,          "t": 5},
        },
    },
    {
        "name":       "moons",
        "X":          X_moons,
        "y_true":     y_moons,
        "n_clusters": 2,
        "r_values":   [2, 4, 7, 12, 25, 50],
        "configs": {
            "A_full_noDiff": {"sigma": 0.1, "k_neighbours": n_moons - 1, "t": 0},
            "C_full_Diff":   {"sigma": 0.1, "k_neighbours": n_moons - 1, "t": 5},
            "D_knn_noDiff":  {"sigma": 0.1, "k_neighbours": 50,          "t": 0},
            "B_knn_Diff":    {"sigma": 0.1, "k_neighbours": 50,          "t": 5},
        },
    },
]

total_runs = sum(
    len(ds["configs"]) * 2 * len(ds["r_values"])
    for ds in experiment_datasets
)

print(f"Experiment overview: {total_runs} total ALC runs")
for ds in experiment_datasets:
    n_runs_ds = len(ds["configs"]) * 2 * len(ds["r_values"])
    print(f"  {ds['name']:10s}  n={ds['X'].shape[0]}  "
          f"configs={len(ds['configs'])}  r_values={ds['r_values']}  "
          f"→ {n_runs_ds} runs")

In [ ]:
import time

results = []
np.random.seed(42)   # seed ALC's stochastic node selection for reproducibility

total_runs = sum(
    len(ds["configs"]) * 2 * len(ds["r_values"])
    for ds in experiment_datasets
)
run_idx = 0
t_start = time.time()

for ds in experiment_datasets:
    for config_name, cfg in ds["configs"].items():
        for lap in ["RW", "SYM"]:
            for r in ds["r_values"]:
                run_idx += 1
                Y, evals = diffusion_spectral_embed(
                    ds["X"],
                    sigma=cfg["sigma"],
                    k_neighbours=cfg["k_neighbours"],
                    n_clusters=ds["n_clusters"],
                    q=r,
                    Laplacian=lap,
                    t=cfg["t"],
                    seed=EMBED_SEED,
                )

                np.random.seed(42)   # re-seed before each ALC call for consistency
                alc    = ALCAdapter()
                labels = alc.fit_predict(Y, cn=None)

                ari     = adjusted_rand_score(ds["y_true"], labels)
                k_found = len(np.unique(labels))

                results.append({
                    "dataset":          ds["name"],
                    "config":           config_name,
                    "laplacian":        lap,
                    "sigma":            cfg["sigma"],
                    "k_neighbours":     cfg["k_neighbours"],
                    "t":                cfg["t"],
                    "rank_r":           r,
                    "ARI":              round(ari, 4),
                    "n_clusters_found": k_found,
                    "delta_k":          abs(k_found - ds["n_clusters"]),
                })

                elapsed = time.time() - t_start
                print(
                    f"  [{run_idx:3d}/{total_runs}]  "
                    f"{ds['name']:8s}  {config_name:15s}  {lap}  r={r:2d}"
                    f"  ARI={ari:.3f}  k={k_found}"
                    f"  ({elapsed:.0f}s elapsed)"
                )

df = pd.DataFrame(results)
print(f"\nDone — {len(df)} rows collected in {time.time() - t_start:.1f}s")

In [ ]:
# Plot 1 — ARI vs Rank for all datasets
config_styles = {
    "A_full_noDiff": {"color": "blue", "linestyle": "-",  "label": "A: Full + No Diff"},
    "C_full_Diff":   {"color": "blue", "linestyle": "--", "label": "C: Full + Diff"},
    "D_knn_noDiff":  {"color": "red",  "linestyle": "-",  "label": "D: k-NN + No Diff"},
    "B_knn_Diff":    {"color": "red",  "linestyle": "--", "label": "B: k-NN + Diff"},
}

for ds_info in experiment_datasets:
    ds_name         = ds_info["name"]
    n_clusters_true = ds_info["n_clusters"]
    sub             = df[df["dataset"] == ds_name]

    fig, axes = plt.subplots(1, 2, figsize=(13, 4), sharey=True)
    fig.suptitle(
        f"ARI vs Rank — {ds_name.capitalize()}  "
        f"(vertical dashed = true q = {n_clusters_true})",
        fontsize=12,
    )

    for ax, lap in zip(axes, ["RW", "SYM"]):
        lap_sub = sub[sub["laplacian"] == lap]
        for cfg_name, style in config_styles.items():
            cfg_sub = lap_sub[lap_sub["config"] == cfg_name].sort_values("rank_r")
            if cfg_sub.empty:
                continue
            ax.plot(
                cfg_sub["rank_r"], cfg_sub["ARI"],
                color=style["color"], linestyle=style["linestyle"],
                marker="o", markersize=4, label=style["label"],
            )
        ax.axvline(
            x=n_clusters_true, color="grey", linestyle="--", alpha=0.6,
            label=f"True q = {n_clusters_true}",
        )
        ax.set_xlabel("Rank r")
        ax.set_ylabel("ARI")
        ax.set_ylim(-0.05, 1.05)
        ax.set_title(f"Laplacian = {lap}")
        ax.legend(fontsize=8)

    plt.tight_layout()
    plt.show()

In [ ]:
# Plot 2 — Cluster count vs Rank for all datasets
for ds_info in experiment_datasets:
    ds_name         = ds_info["name"]
    n_clusters_true = ds_info["n_clusters"]
    sub             = df[df["dataset"] == ds_name]

    fig, axes = plt.subplots(1, 2, figsize=(13, 4), sharey=True)
    fig.suptitle(
        f"Cluster Count vs Rank — {ds_name.capitalize()}  "
        f"(horizontal dashed = true k = {n_clusters_true})",
        fontsize=12,
    )

    for ax, lap in zip(axes, ["RW", "SYM"]):
        lap_sub = sub[sub["laplacian"] == lap]
        for cfg_name, style in config_styles.items():
            cfg_sub = lap_sub[lap_sub["config"] == cfg_name].sort_values("rank_r")
            if cfg_sub.empty:
                continue
            ax.plot(
                cfg_sub["rank_r"], cfg_sub["n_clusters_found"],
                color=style["color"], linestyle=style["linestyle"],
                marker="o", markersize=4, label=style["label"],
            )
        ax.axhline(
            y=n_clusters_true, color="grey", linestyle="--", alpha=0.6,
            label=f"True k = {n_clusters_true}",
        )
        ax.set_xlabel("Rank r")
        ax.set_ylabel("Clusters found")
        ax.set_title(f"Laplacian = {lap}")
        ax.legend(fontsize=8)

    plt.tight_layout()
    plt.show()

In [ ]:
def _scatter_grid(ds_info, r_val):
    """
    Render a 2 × 2 grid of embedding scatter plots (first two dims of Y)
    for all four configurations at a given rank r_val.
    Works for any dataset, colouring points by ground truth label.
    """
    unique_lbl = np.unique(ds_info["y_true"])
    n_labels   = len(unique_lbl)
    colours    = plt.cm.tab10(np.linspace(0, 0.9, n_labels))

    config_layout = [
        ("A_full_noDiff", 0, 0, "A: Full + No Diff"),
        ("C_full_Diff",   0, 1, "C: Full + Diff"),
        ("D_knn_noDiff",  1, 0, "D: k-NN + No Diff"),
        ("B_knn_Diff",    1, 1, "B: k-NN + Diff"),
    ]

    fig, axes = plt.subplots(2, 2, figsize=(10, 8), constrained_layout=True)
    fig.suptitle(
        f"{ds_info['name'].capitalize()} — Embedding (dims 1 & 2), "
        f"r = {r_val},  Laplacian = RW",
        fontsize=12,
    )

    for config_name, row, col, title in config_layout:
        cfg = ds_info["configs"][config_name]
        Y, _ = diffusion_spectral_embed(
            ds_info["X"],
            sigma=cfg["sigma"],
            k_neighbours=cfg["k_neighbours"],
            n_clusters=ds_info["n_clusters"],
            q=r_val,
            Laplacian="RW",
            t=cfg["t"],
            seed=EMBED_SEED,
        )
        ax = axes[row, col]
        for i, lbl in enumerate(unique_lbl):
            mask = ds_info["y_true"] == lbl
            ax.scatter(Y[mask, 0], Y[mask, 1], s=8, alpha=0.7,
                       color=colours[i], label=f"Class {lbl}")
        ax.set_title(title, fontsize=10)
        ax.set_xlabel("Dim 1")
        ax.set_ylabel("Dim 2")
        ax.legend(markerscale=2, fontsize=7)

    plt.show()

In [ ]:
# Plot 3 — canonical rank across all three datasets
print("Plot 3 — Embeddings at canonical rank (r = n_clusters per dataset)")
print("=" * 60)
for ds_info in experiment_datasets:
    r_val = ds_info["n_clusters"]
    print(f"\n  {ds_info['name'].capitalize()}  (r = {r_val})")
    _scatter_grid(ds_info, r_val=r_val)

In [ ]:
# Plot 4 — heavy over-specification (r = 25) across all three datasets
print("Plot 4 — Embeddings at r = 25 (over-specified)")
print("=" * 50)
for ds_info in experiment_datasets:
    print(f"\n  {ds_info['name'].capitalize()}")
    _scatter_grid(ds_info, r_val=25)

In [ ]:
q_lookup = {"blobs": 7, "circles": 3, "moons": 2}
col_order = ["A_full_noDiff", "C_full_Diff", "D_knn_noDiff", "B_knn_Diff"]

for ds_name in ["blobs", "circles", "moons"]:
    q_true = q_lookup[ds_name]
    print(f"\n{'=' * 60}")
    print(f"  {ds_name.upper()}  —  ARI at r = {q_true} (canonical) and r = 25 (over-specified)")
    print(f"{'=' * 60}")

    subset = df[
        (df["dataset"] == ds_name) &
        (df["rank_r"].isin([q_true, 25]))
    ]

    table = subset.pivot_table(
        index=["laplacian", "rank_r"],
        columns="config",
        values="ARI",
    ).round(3)

    present_cols = [c for c in col_order if c in table.columns]
    print(table[present_cols].to_string())

---

## Interpretation of Step 4.3 Results

*Fill in after running the cells above.*

---

**Q1 — Does k-NN alone (D vs A) improve recovery on circles and moons?**

*Look at the vertical gap between A (blue solid) and D (red solid) at r = q in Plot 1.*

Answer:

---

**Q2 — Does diffusion alone (C vs A) improve recovery on circles and moons?**

*Look at the gap between A (blue solid) and C (blue dashed) at r = q and r = 25 in Plot 1.*

Answer:

---

**Q3 — Does diffusion add value on top of k-NN (B vs D)?**

*Look at the gap between D (red solid) and B (red dashed) at r = 25 in Plot 1.*

- If B ≈ D at all ranks: diffusion adds negligible value; the pre-print should emphasise the k-NN contribution.
- If B > D at high ranks: both modifications contribute; the pre-print can make a stronger joint claim.

Answer: